# Evidence audit: success-guided selection
Reads existing Drive artifacts. **No training, policy inference, new rollouts, or changes to experiment directories.** CPU runtime is enough. Writes only a separate evidence directory and a local dataset cache.

Questions: Does the classifier rank successful trajectories? Does it prefer easy or repetitive demonstrations? On paired starts, where do random and success-guided policies differ?

All analyses are exploratory. Classifier validation selected its checkpoint, so it is not an independent test. Final policy outcomes already informed this redesign. Associations cannot establish why selection failed. This notebook does not retrofit a better-looking success threshold.


In [ ]:
from pathlib import Path
import sys, subprocess, json, os
ROOT = Path('/content/robotics-data-flywheel-evidence')
REPOSITORY = 'https://github.com/0-5-blood-prince/robotics-data-flywheel.git'
if not ROOT.exists():
    subprocess.run(['git','clone',REPOSITORY,str(ROOT)],check=True)
# Keep an existing checkout unchanged. Only read helpers and saved artifacts.
subprocess.run([sys.executable,'-m','pip','install','-e',str(ROOT),
                'pandas','matplotlib','scikit-learn','zarr<3'],check=True)
sys.path.insert(0,str(ROOT/'src'))
import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
from IPython.display import display, FileLink, Video
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, roc_curve
from robot_flywheel.data import encode
from robot_flywheel.demo_score import classifier
from robot_flywheel.prepare import prepare
from robot_flywheel.artifacts import digest
from google.colab import drive
drive.mount('/content/drive')
WORK = Path('/content/drive/MyDrive/robotics-data-flywheel-runs')
ORIGINAL = WORK/'exploratory_ddpm_v1_seed11'
GUIDED = WORK/'success_guided_v3_seed11'
EVIDENCE = WORK/'success_guided_v3_evidence'
for experiment in (ORIGINAL,GUIDED):
    if EVIDENCE.resolve()==experiment.resolve() or experiment.resolve() in EVIDENCE.resolve().parents:
        raise ValueError('Evidence must be outside the experiment directories')
EVIDENCE.mkdir(parents=True,exist_ok=True)
print('Read:',ORIGINAL,GUIDED,'\nWrite:',EVIDENCE)


## 1. Inventory and provenance
Fail early if required files are missing. Load only your own trusted checkpoints. Confirm paired evaluation settings and initial states before comparing outcomes.

In [ ]:
import hashlib

def read_json(path):
    return json.loads(Path(path).read_text())
def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
    return h.hexdigest()
def save_json(name,value):
    (EVIDENCE/name).write_text(json.dumps(value,indent=2,allow_nan=False))
def save_table(name,table):
    table.to_csv(EVIDENCE/name,index=False)
    display(table)
def save_fig(name):
    plt.gcf().savefig(EVIDENCE/name,dpi=150,bbox_inches='tight');plt.show()

old_result=read_json(ORIGINAL/'results.json');new_result=read_json(GUIDED/'results.json')
contract=read_json(GUIDED/'classifier_contract.json')
protocol=read_json(GUIDED/'frozen_protocol.json')
assert old_result['protocol_hash']==contract['original_protocol']
assert new_result['protocol_hash']==protocol['sha256']
SEED=protocol['protocol']['training_seeds'][0]
ROUNDS=protocol['protocol']['rounds']
assert len(protocol['protocol']['training_seeds'])==1
records=old_result['records']+new_result['records']
assert {r['seed'] for r in records}=={SEED}
# Resolve paths relative to the selected run folders (also permits moved Drive folders).
def stage_path(record):
    home=GUIDED if record['arm']=='success_guided' else ORIGINAL
    stage=home/f'seed_{record["seed"]}'/record['arm']
    return stage if record['arm']=='baseline' else stage/f'round_{record["round"]}'

summaries={}; initial_states={}; inventory=[]
expected=list(range(protocol['protocol']['evaluation_start'],
                    protocol['protocol']['evaluation_start']+protocol['protocol']['evaluation_count']))
for r in records:
    directory=stage_path(r)/'final_evaluation'
    s=read_json(directory/'summary.json')
    assert s['seeds']==expected
    assert s['sampler']=='ddpm' and s['execute']==protocol['protocol']['execute']
    assert s['max_steps']==protocol['protocol']['max_steps']
    assert s['checkpoint']==sha(stage_path(r)/'latest.pt')
    for row in s['rows']:
        if row['seed'] in initial_states:
            assert np.allclose(initial_states[row['seed']],row['initial_state'],atol=1e-7,rtol=0)
        initial_states[row['seed']]=row['initial_state']
    summaries[r['arm'],r['round']]=s
    inventory.append(dict(arm=r['arm'],round=r['round'],episodes=r['episodes'],
                          success_rate=s['success_rate'],mean_max_coverage=s['mean_max_coverage']))
save_table('policy_inventory.csv',pd.DataFrame(inventory))
saved=torch.load(GUIDED/'classifier.pt',map_location='cpu',weights_only=False)
assert saved['contract_hash']==digest(contract)
model=classifier(saved['dropout']);model.load_state_dict(saved['state_dict']);model.eval()
mean=np.asarray(saved['mean']);std=np.asarray(saved['std'])
def state_scores(states):
    x=(encode(states)-mean)/std
    with torch.no_grad():return model(torch.tensor(x,dtype=torch.float32)).sigmoid().numpy().ravel()
save_json('provenance.json',dict(classifier_sha256=sha(GUIDED/'classifier.pt'),
    classifier_contract=contract,original_protocol=old_result['protocol_hash'],
    guided_protocol=new_result['protocol_hash'],analysis='Exploratory; validation used for checkpoint selection'))
print('Artifact identities and paired physical starts verified.')


## 2. Classifier discrimination and possible outcome shortcuts
Score each saved rollout by its **mean state probability**, matching demonstration ranking. Compare that with initial-state-only, first-10-state, and last-10-state scores. A signal concentrated near the end may reflect task progress instead of an informative strategy; it is evidence to investigate, not proof.

ROC-AUC: 0.5 is chance ranking. Average precision (AP) is compared with the success prevalence. Confidence intervals resample whole trajectories, never individual frames. These are exploratory intervals on the checkpoint-selection split. Scores need not be calibrated outcome probabilities.


In [ ]:
rollout_rows=[]
for index,split in enumerate(['training','validation']):
    directory=GUIDED/f'classifier_rollouts_{index}'
    summary=read_json(directory/'summary.json')
    assert summary['checkpoint']==contract['checkpoints'][index]
    for row in summary['rows']:
        with np.load(directory/f"episode_{row['seed']}.npz") as trace:
            states=trace['states'];coverage=trace['coverage']
            scores=state_scores(states)
        rollout_rows.append(dict(split=split,seed=row['seed'],success=int(row['success']),
            score=float(scores.mean()),initial_score=float(scores[0]),
            prefix_score=float(scores[:10].mean()),tail_score=float(scores[-10:].mean()),
            steps=len(scores),max_coverage=float(np.max(coverage)),final_coverage=float(coverage[-1]),
            initial_goal_distance=float(np.linalg.norm(states[0,2:4]-[256,256]))))
rollouts=pd.DataFrame(rollout_rows)
rollouts.to_csv(EVIDENCE/'classifier_rollouts.csv',index=False)

def ranking_metrics(y,s,bootstrap=300):
    y=np.asarray(y);s=np.asarray(s)
    result=dict(episodes=len(y),successes=int(y.sum()),prevalence=float(y.mean()))
    if len(np.unique(y))<2:
        return dict(**result,auc=None,ap=None,auc_low=None,auc_high=None,ap_low=None,ap_high=None)
    rng=np.random.default_rng(17);boots=[]
    for _ in range(bootstrap):
        ix=rng.integers(0,len(y),len(y))
        if len(np.unique(y[ix]))==2:
            boots.append([roc_auc_score(y[ix],s[ix]),average_precision_score(y[ix],s[ix])])
    bounds=np.quantile(boots,[.025,.975],axis=0)
    return dict(**result,auc=float(roc_auc_score(y,s)),ap=float(average_precision_score(y,s)),
                auc_low=float(bounds[0,0]),auc_high=float(bounds[1,0]),
                ap_low=float(bounds[0,1]),ap_high=float(bounds[1,1]))
metrics=[]
for split,group in rollouts.groupby('split'):
    for score in ['score','initial_score','prefix_score','tail_score']:
        metrics.append(dict(split=split,feature=score,**ranking_metrics(group.success,group[score])))
metrics=pd.DataFrame(metrics)
save_table('classifier_metrics.csv',metrics)
display(saved['report'] if 'history' not in saved['report'] else {k:v for k,v in saved['report'].items() if k!='history'})
v=rollouts.query("split=='validation'")
fig,axs=plt.subplots(1,3,figsize=(15,4))
for outcome,label in [(0,'Failure'),(1,'Success')]:
    axs[0].hist(v.loc[v.success==outcome,'score'],bins=15,alpha=.5,label=label)
axs[0].set(xlabel='Mean classifier score',ylabel='Rollouts');axs[0].legend()
if v.success.nunique()==2:
    for feature in ['score','initial_score','prefix_score','tail_score']:
        fpr,tpr,_=roc_curve(v.success,v[feature]);axs[1].plot(fpr,tpr,label=feature)
        precision,recall,_=precision_recall_curve(v.success,v[feature]);axs[2].plot(recall,precision,label=feature)
axs[1].plot([0,1],[0,1],'k--');axs[1].set(xlabel='False positive rate',ylabel='True positive rate');axs[1].legend()
axs[2].axhline(v.success.mean(),color='gray',linestyle='--');axs[2].set(xlabel='Recall',ylabel='Precision');axs[2].legend()
save_fig('classifier_discrimination.png')
# Descriptive correlations, not independent statistical tests or causal effects.
save_table('classifier_correlations.csv',v[['score','initial_score','steps','max_coverage','final_coverage','initial_goal_distance']]
           .corr(method='spearman').rename_axis('feature').reset_index())


## 3. What demonstrations were acquired?
Load the original expert data and verify its archive identity. Compare **new acquisitions only** at each round, rather than letting the common initial 50 episodes conceal differences.

Distances below use the project's geometric features, so their scale is a design choice. Initial pose spread and nearest-state distance are descriptive proxies, not measures of actual skill or expert success. Terminal goal distance is not a completion label. Scoring every demonstration also lets us reproduce the saved ranking.


In [ ]:
EPISODES,SOURCE=prepare(ROOT/'data/pusht')
original_frozen=read_json(ORIGINAL/'frozen_protocol.json')
assert SOURCE['archive_sha256']==original_frozen['source_sha256']
from robot_flywheel.selection import geometry,representatives
by_id={e.id:e for e in EPISODES}
base=torch.load(ORIGINAL/f'seed_{SEED}'/'baseline/latest.pt',map_location='cpu',weights_only=False)
initial=set(base['manifest']['episode_ids'])
initial_states=np.concatenate([representatives(by_id[i]) for i in initial])
features=[]
for ep in EPISODES:
    s=ep.states; rep=representatives(ep)
    distance=np.sqrt(((rep[:,None,:]-initial_states[None,:,:])**2).sum(-1).min(1))
    features.append(dict(episode_id=ep.id,initial=ep.id in initial,score=float(state_scores(s).mean()),
        length=len(s),initial_x=float(s[0,2]),initial_y=float(s[0,3]),initial_angle=float(s[0,4]),
        initial_goal_distance=float(np.linalg.norm(s[0,2:4]-[256,256])),
        terminal_goal_distance=float(np.linalg.norm(s[-1,2:4]-[256,256])),
        object_path_length=float(np.linalg.norm(np.diff(s[:,2:4],axis=0),axis=1).sum()),
        mean_distance_to_initial=float(distance.mean())))
features=pd.DataFrame(features).set_index('episode_id')
ranking=read_json(GUIDED/'ranking.json')
assert all(np.isclose(features.loc[r['episode_id'],'score'],r['score'],atol=1e-5) for r in ranking)
features.to_csv(EVIDENCE/'expert_features.csv')
selections={};selection_rows=[]
for arm,home in [('random',ORIGINAL),('targeted',ORIGINAL),('diversity',ORIGINAL),('success_guided',GUIDED)]:
    acquired=set(initial)
    for r in range(1,ROUNDS+1):
        selection=read_json(home/f'seed_{SEED}'/arm/f'round_{r}'/'selection.json')
        ids=selection['added_ids']
        assert not acquired.intersection(ids) and len(set(ids))==len(ids)
        acquired.update(ids)
        assert acquired==set(selection['cumulative_ids'])
        selections[arm,r]=ids
        poses=np.stack([geometry(by_id[i].states[0]) for i in ids])
        pair=np.sqrt(((poses[:,None,:]-poses[None,:,:])**2).sum(-1))
        upper=pair[np.triu_indices(len(ids),1)]
        nearest=pair+np.eye(len(ids))*1e9
        group=features.loc[ids]
        selection_rows.append(dict(arm=arm,round=r,added=len(ids),transitions=int(group.length.sum()),
            mean_score=float(group.score.mean()),mean_length=float(group.length.mean()),
            mean_initial_goal_distance=float(group.initial_goal_distance.mean()),
            mean_terminal_goal_distance=float(group.terminal_goal_distance.mean()),
            mean_state_novelty=float(group.mean_distance_to_initial.mean()),
            mean_initial_pair_distance=float(upper.mean()) if len(upper) else None,
            mean_initial_nearest_distance=float(nearest.min(1).mean()) if len(ids)>1 else None,
            overlap_with_random=len(set(ids)&set(selections.get(('random',r),[])))))
selection_stats=pd.DataFrame(selection_rows)
save_table('selection_diagnostics.csv',selection_stats)
fig,axs=plt.subplots(1,3,figsize=(15,4))
for arm in ['random','targeted','diversity','success_guided']:
    ids=sum([selections[arm,r] for r in range(1,ROUNDS+1)],[])
    g=features.loc[ids]
    axs[0].scatter(g.initial_x,g.initial_y,s=18,alpha=.6,label=arm)
    axs[1].hist(g.initial_angle,bins=np.linspace(0,2*np.pi,13),alpha=.35,label=arm)
    rows=selection_stats[selection_stats.arm==arm]
    axs[2].plot(rows['round'],rows.mean_state_novelty,marker='o',label=arm)
axs[0].set(xlabel='Initial object x',ylabel='Initial object y',aspect='equal');axs[0].legend()
axs[1].set(xlabel='Initial object angle (radians)',ylabel='Acquired episodes')
axs[2].set(xlabel='Round',ylabel='Mean distance to initial dataset');axs[2].legend()
save_fig('acquisition_geometry.png')
candidate_features=features.loc[~features.initial]
save_table('expert_score_correlations.csv',candidate_features[['score','length','initial_goal_distance','terminal_goal_distance','object_path_length','mean_distance_to_initial']]
           .corr(method='spearman').rename_axis('feature').reset_index())


## 4. Paired policy outcomes and failure signatures
Compare final-round policies on exactly the same initial states. Report all four outcome combinations, not just a favorable example. Coverage summaries distinguish maximum progress from final progress. A large drop is a descriptive signature; it does not diagnose its physical cause.

Below, “near goal” uses max coverage ≥0.90 solely for descriptive failure grouping. Environment success labels remain unchanged. Counts are not independent training replicates.


In [ ]:
paired=[]
def final_directory(arm):
    home=GUIDED if arm=='success_guided' else ORIGINAL
    return home/f'seed_{SEED}'/arm/f'round_{ROUNDS}'/'final_evaluation'
for arm in ['random','success_guided']:
    s=summaries[arm,ROUNDS]
    for row in s['rows']:
        with np.load(final_directory(arm)/f"episode_{row['seed']}.npz") as trace:
            cov=trace['coverage']
        assert len(cov)>0 and np.isclose(cov.max(),row['max_coverage'])
        paired.append(dict(arm=arm,seed=row['seed'],success=bool(row['success']),
            max_coverage=float(cov.max()),final_coverage=float(cov[-1]),
            coverage_drop=float(cov.max()-cov[-1]),steps=len(cov),
            failure_group='success' if row['success'] else ('near_goal_failure' if cov.max()>=.90 else 'lower_coverage_failure')))
paired=pd.DataFrame(paired);paired.to_csv(EVIDENCE/'paired_rollout_metrics.csv',index=False)
wide=paired.pivot(index='seed',columns='arm',values='success')
wide['category']=np.select([wide.random & wide.success_guided,wide.random & ~wide.success_guided,
                           ~wide.random & wide.success_guided],
                          ['both_success','random_only','success_guided_only'],default='both_fail')
save_table('paired_outcomes.csv',wide.reset_index())
display(wide.category.value_counts().reindex(['both_success','random_only','success_guided_only','both_fail'],fill_value=0))
save_table('failure_groups.csv',paired.groupby(['arm','failure_group']).size().rename('count').reset_index())
save_table('coverage_diagnostics.csv',paired.groupby('arm')[['max_coverage','final_coverage','coverage_drop','steps']].mean().reset_index())
fig,axs=plt.subplots(1,2,figsize=(11,4))
for arm,g in paired.groupby('arm'):
    axs[0].hist(g.max_coverage,bins=np.linspace(0,1,21),alpha=.5,label=arm)
    axs[1].hist(g.coverage_drop,bins=np.linspace(0,1,21),alpha=.5,label=arm)
axs[0].set(xlabel='Maximum coverage',ylabel='Rollouts');axs[0].legend()
axs[1].set(xlabel='Maximum minus final coverage',ylabel='Rollouts');axs[1].legend()
save_fig('paired_coverage.png')


## 5. Deterministic example inspection
Select the lowest two seed IDs in each disagreement group, plus one shared failure. This makes the selection rule explicit. Plots use stored states and coverage: no physics replay or hidden simulation. Object paths alone cannot prove collision/contact causes.

Videos are shown only if already saved for that exact seed. Missing videos do not trigger new rollouts. Change `EXAMPLES_PER_GROUP` to inspect more stored traces.


In [ ]:
EXAMPLES_PER_GROUP=2
example_ids=[]
for category,count in [('random_only',EXAMPLES_PER_GROUP),('success_guided_only',EXAMPLES_PER_GROUP),('both_fail',1)]:
    ids=sorted(wide.index[wide.category==category])[:count]
    for seed in ids:
        example_ids.append(dict(seed=int(seed),category=category))
        fig,axs=plt.subplots(1,3,figsize=(14,4))
        for ax,arm in zip(axs[:2],['random','success_guided']):
            directory=final_directory(arm)
            with np.load(directory/f'episode_{seed}.npz') as trace:
                states=np.concatenate([trace['states'][:1],trace['next_states']],axis=0)
                cov=trace['coverage']
            ax.plot(states[:,0],states[:,1],alpha=.6,label='Agent')
            ax.plot(states[:,2],states[:,3],label='Object')
            ax.scatter(*states[0,2:4],marker='o',label='Object start')
            ax.scatter(*states[-1,2:4],marker='x',label='Object end')
            ax.scatter(256,256,marker='+',s=100,label='Goal center')
            ax.set(xlim=(0,512),ylim=(0,512),aspect='equal',title=arm);ax.legend(fontsize=8)
            axs[2].plot(np.arange(1,len(cov)+1),cov,label=arm)
        axs[2].set(xlabel='Control step',ylabel='Coverage',ylim=(0,1));axs[2].legend()
        fig.suptitle(f'Seed {seed}: {category}');save_fig(f'paired_example_{seed}.png')
        for arm in ['random','success_guided']:
            video=final_directory(arm)/f'episode_{seed}.mp4'
            if video.exists():
                print(arm);display(Video(str(video),embed=True,width=360))
            else:print(f'{arm}, seed {seed}: no saved video; stored traces shown above.')
save_json('example_selection.json',example_ids)


## 6. Evidence summary and export
Use the measurements to choose the next experiment. Weak validation discrimination suggests stopping this selector; unusually low novelty suggests testing diversity mixing; widespread near-goal failures suggest investigating policy execution and demonstrated completion. None of these observations establishes the cause or guarantees a fix.

Keep the original results and this report. Further model/selector decisions should use development starts; use fresh starts and more training seeds for confirmation. No all-data reference is trained by this notebook.


In [ ]:
m=metrics[(metrics.split=='validation') & (metrics.feature=='score')].iloc[0]
counts=wide.category.value_counts()
summary=f"""# Saved-artifact evidence report

Exploratory analysis, one training seed. No new policy training or simulation.

- Validation trajectories: {int(m.episodes)}; successes: {int(m.successes)}.
- Trajectory score ROC-AUC: {m.auc:.3f} (bootstrap interval {m.auc_low:.3f} to {m.auc_high:.3f}).
- Average precision: {m.ap:.3f}; prevalence reference: {m.prevalence:.3f}.
- Random-only successes: {counts.get('random_only',0)}.
- Success-guided-only successes: {counts.get('success_guided_only',0)}.
- Both succeed: {counts.get('both_success',0)}; both fail: {counts.get('both_fail',0)}.

See classifier_metrics.csv for prefix/tail comparisons, selection_diagnostics.csv for acquisition diversity, and paired_rollout_metrics.csv for progress loss. These associations do not establish causality. Classifier validation was used for checkpoint selection, so it is not independent test evidence. Original evaluation results informed the redesign.
"""
(EVIDENCE/'evidence_report.md').write_text(summary)
print(summary)
for path in sorted(EVIDENCE.iterdir()):
    if path.is_file():display(FileLink(str(path)))
